# CNOT and entanglement
**Quantum Computing · Week 4 (Entanglement) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qc/w4d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

The CNOT gate flips a target qubit when a control qubit is 1. Put an H before it and you get a
**Bell state**: two qubits whose outcomes always agree, although each one alone is a fair coin.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def probs(sv):
    return {str(k): round(float(v), 3) for k, v in sv.probabilities_dict().items()}

cnot = QuantumCircuit(2)
cnot.cx(0, 1)                       # control = qubit 0, target = qubit 1

print("CNOT truth table (labels are q1 q0):")
for label in ["00", "01", "10", "11"]:
    out = Statevector.from_label(label).evolve(cnot)
    print(f"  |{label}>  ->  |{list(probs(out))[0]}>")

Only the inputs where qubit 0 (the right-hand bit) is 1 get their qubit 1 flipped.

## The Bell state

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

sv = Statevector(bell)
print("amplitudes (00, 01, 10, 11):", np.round(sv.data.real, 3))
print("probabilities:", probs(sv))
bell.draw("mpl", style="clifford")

In [ ]:
from qiskit.primitives import StatevectorSampler

measured = bell.copy()
measured.measure_all()
result = StatevectorSampler(seed=3).run([measured], shots=1000).result()[0]
print("counts:", result.data.meas.get_counts())
print("first 16 shots:", " ".join(result.data.meas.get_bitstrings()[:16]))

Never `01` or `10`: the two qubits always agree. Yet each qubit on its own is random.

## Is it a product state?
Two separate qubits always give amplitudes with $a_{00}\,a_{11} - a_{01}\,a_{10} = 0$.
A state that breaks this rule cannot be split into two separate qubits: it is **entangled**.

In [ ]:
def entanglement_test(sv):
    a00, a01, a10, a11 = sv.data
    return abs(a00 * a11 - a01 * a10)

separate = QuantumCircuit(2)
separate.h(0)                        # H on qubit 0 only, no CNOT
print("H only     :", round(entanglement_test(Statevector(separate)), 3), "-> product state")
print("Bell state :", round(entanglement_test(Statevector(bell)), 3), "-> entangled")

## One qubit of an entangled pair
`partial_trace` keeps qubit 0 and "forgets" qubit 1. What's left is a *mixed* state: its Bloch vector
has shrunk to the center of the sphere. (Mixed states are week 5's topic.)

In [ ]:
from qiskit.quantum_info import partial_trace

rho0 = partial_trace(Statevector(bell), [1])          # trace out qubit 1
print("density matrix of qubit 0 alone:")
print(np.round(rho0.data.real, 3))
print("purity:", round(float(np.real(rho0.purity())), 3), "  (1 = pure state, 0.5 = completely mixed)")

## Your turn: the CNOT truth table
Apply `cx(0, 1)` (control qubit 0, target qubit 1) to each of the four basis states and print where each one goes.
Then do the same with the roles swapped, `cx(1, 0)`. Remember: labels read q1 q0.

In [ ]:
from qiskit.quantum_info import Statevector

for label in ["00", "01", "10", "11"]:
    qc = QuantumCircuit(2)
    # TODO: add the CNOT with control 0 and target 1
    out = Statevector.from_label(label).evolve(qc)
    (result,) = out.probabilities_dict().keys()      # a basis state goes to a single basis state
    print(f"|{label}> -> |{result}>")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for control, target in [(0, 1), (1, 0)]:
    print(f"cx({control}, {target}): control q{control}, target q{target}")
    for label in ["00", "01", "10", "11"]:
        qc = QuantumCircuit(2)
        qc.cx(control, target)
        out = Statevector.from_label(label).evolve(qc)
        (result,) = out.probabilities_dict().keys()
        print(f"   |{label}> -> |{result}>")
print("The target flips exactly when the control is 1. With q0 as control, that's when the RIGHT character is 1.")